# M16 lab solution: Retrievers & Hybrid Search

**Brief.** Hybrid upgrade: add BM25 to the vector retriever, tune the fusion weights on the golden set, and find three queries that only hybrid search answers.

Run with the default environment: `uv run jupyter lab`.

Open this notebook from `lab/solution/`, so `../data/` resolves to the lab's own data folder.

## Setup

In [ ]:
import json
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # API keys from the .env file at the repo root
sys.path.insert(0, str(Path("../../../../src").resolve()))

from lab_kit import baseline, hit_rate, is_hit, load_handbook, load_jsonl, mrr, recall_at_k  # noqa: E402

DATA = Path("../data/larkspur")
handbook = load_handbook(DATA / "handbook")
golden = load_jsonl(DATA / "golden.jsonl")
by_id = {g["id"]: g for g in golden}
scored = [g for g in golden if g["evidence_spans"]]
print(len(handbook), "handbook files,", len(golden), "golden questions,", len(scored), "with evidence")

In [ ]:
from helpers import get_embeddings, get_experientiallabs_llm

llm = get_experientiallabs_llm()
embeddings = get_embeddings()
LLM_NAME = llm.model_name
EMBEDDING_NAME = getattr(embeddings, "model", None) or getattr(embeddings, "endpoint", None)
print("LLM:", LLM_NAME, "| embeddings:", EMBEDDING_NAME)

In [ ]:
# The inline "Part V system", reused by the later labs: the handbook split as lab_kit.baseline splits it
# (recursive, 500 characters, 50 overlap), a FAISS vector retriever over those chunks, and an answer
# prompt that cites numbered passages and is allowed to say "I don't know".
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [
    Document(page_content=c, metadata={"source": p["source"], "section": p["title"]})
    for p in handbook
    for c in splitter.split_text(p["text"])
]
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You answer questions from Larkspur Cloud customers using only the numbered handbook passages. "
               "Cite the passages you use as [1], [2]. If the passages do not contain the answer, "
               "reply exactly: I don't know."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{i}] ({d.metadata.get('source', '?')}) {d.page_content}" for i, d in enumerate(docs, 1))


def answer(question, docs, config=None):
    return (ANSWER_PROMPT | llm).invoke({"context": format_docs(docs), "question": question}, config=config).content


print(len(chunks), "chunks indexed")

In [ ]:
def texts(docs):
    return [d.page_content for d in docs]


def score(results, questions=None):
    """results: question id -> retrieved chunk texts in rank order; scored with lab_kit on the questions with evidence."""
    qs = questions or golden
    return {"hit_rate": round(hit_rate(results, qs), 3), "recall_at_k": round(recall_at_k(results, qs), 3),
            "mrr": round(mrr(results, qs), 3)}

## 1. BM25, dense and their fusion

`BM25Retriever` scores shared tokens; the FAISS retriever compares embeddings. `EnsembleRetriever` fuses the two ranked lists with weighted Reciprocal Rank Fusion: each list adds `weight / (60 + rank)` for every chunk it returns. Each retriever returns its top 10 and the fused list is cut to the top 4, the context every row of this lab sends.

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

K, DEPTH = 4, 10
bm25 = BM25Retriever.from_documents(chunks, k=DEPTH)
dense = vectorstore.as_retriever(search_kwargs={"k": DEPTH})


def hybrid(weight_bm25):
    return EnsembleRetriever(retrievers=[bm25, dense], weights=[weight_bm25, 1 - weight_bm25])


def run(search, questions):
    return {g["id"]: texts(search(g["question"]))[:K] for g in questions}

## 2. Sweep the fusion weight

The BM25 weight goes from 0 (dense only) to 1 (BM25 only) in steps of 0.1. The best weight is the one with the highest MRR on the 25 questions with evidence, ties broken by hit rate.

In [ ]:
sweep = []
for w in [round(x * 0.1, 1) for x in range(11)]:
    sweep.append({"bm25_weight": w, **score(run(hybrid(w).invoke, golden))})
    print(sweep[-1])

import pandas as pd

sweep_df = pd.DataFrame(sweep)
best = sweep_df.sort_values(["mrr", "hit_rate"], ascending=False).iloc[0]
BEST_W = float(best["bm25_weight"])
print("best BM25 weight:", BEST_W, dict(best))

## 3. Questions only hybrid search answers

A question counts when the hybrid top 4 (at the best weight) contains an evidence span and neither BM25 alone nor dense alone does, each at its own top 4. If the golden set gives fewer than three, the five lab-local questions in `../data/extra_questions.jsonl` are scored the same way and the cell says so.

In [ ]:
def only_hybrid(questions):
    found = []
    for g in questions:
        def hit(search):
            return any(is_hit(c, g["evidence_spans"]) for c in texts(search(g["question"]))[:K])
        h, b, d = hit(hybrid(BEST_W).invoke), hit(bm25.invoke), hit(dense.invoke)
        if h and not b and not d:
            found.append({"id": g["id"], "question": g["question"]})
    return found


answered = only_hybrid(scored)
print(len(answered), "golden questions answered only by hybrid search")
if len(answered) < 3:
    extra = load_jsonl(Path("../data/extra_questions.jsonl"))
    extra_found = only_hybrid(extra)
    print("Fewer than three in the golden set; lab-local questions answered only by hybrid search:", len(extra_found))
    answered += [{**e, "id": e["id"] + " (lab-local)"} for e in extra_found]
for a in answered:
    print(a["id"], a["question"])

## Results

In [ ]:
print("run:", time.strftime("%Y-%m-%d"), "| embeddings:", EMBEDDING_NAME)
print(sweep_df.to_string(index=False))
print("\nonly hybrid:", [a["id"] for a in answered])

## What this shows

- The ends of the sweep are the two single retrievers (weight 0 is dense only, weight 1 is BM25 only); the curve between them shows whether fusion beats both.
- RRF uses ranks only, so a chunk ranked fairly high by both retrievers can beat one ranked first by only one of them. The only-hybrid questions are where that happens on the handbook.
- `BM25Retriever` splits on whitespace, so punctuation and case decide matches: a question with an exact product term next to a full stop may miss it. Look at the questions BM25 alone misses before tuning the weight further.